# NYC Taxi Analytics - Training Notebook

**Dataset:** `samples.nyctaxi.trips`  
**Purpose:** Hands-on training for business users  
**Duration:** 40 minutes

---

## Table of Contents
1. Basic Exploration
2. Analytical Queries for Visualizations
3. Dashboard Queries
4. Advanced Analysis (Optional)
5. Data Quality Checks (Optional)

## Section 1: Basic Exploration Queries

Start here to get familiar with the NYC Taxi dataset.

In [0]:
%sql
-- Query 1.1: Preview the data
-- Purpose: Get familiar with the table structure and sample records

SELECT * 
FROM samples.nyctaxi.trips 
LIMIT 10;

In [0]:
%sql
-- Query 1.2: Count total trips
-- Purpose: Understand the dataset size

SELECT COUNT(*) as total_trips
FROM samples.nyctaxi.trips;

In [0]:
%sql
-- Query 1.3: Date range of data
-- Purpose: Know the time period covered

SELECT 
  MIN(DATE(tpep_pickup_datetime)) as earliest_trip,
  MAX(DATE(tpep_pickup_datetime)) as latest_trip
FROM samples.nyctaxi.trips
WHERE tpep_pickup_datetime IS NOT NULL;

## Section 2: Analytical Queries for Visualizations

These queries are designed to create specific visualizations:
- Bar Charts
- Line Charts  
- Scatter Plots
- Tables

In [0]:
%sql
-- Query 2.1: Top Pickup Locations
-- Purpose: Bar Chart / Table - Busiest pickup zones
-- Visualization: Create a BAR CHART with pickup_zip on X-axis and trip_count on Y-axis
-- Save as: "Top Pickup Locations"

SELECT 
  pickup_zip,
  COUNT(*) as trip_count,
  ROUND(AVG(fare_amount), 2) as avg_fare,
  ROUND(SUM(fare_amount), 2) as total_revenue
FROM samples.nyctaxi.trips
WHERE pickup_zip IS NOT NULL
GROUP BY pickup_zip
ORDER BY trip_count DESC
LIMIT 10;

In [0]:
%sql
-- Query 2.2: Trips by Distance Category
-- Purpose: Bar Chart - Trip distribution by distance
-- Visualization: Create a BAR CHART with distance_category on X-axis and trips on Y-axis
-- Save as: "Trips by Distance Category"

SELECT 
  CASE 
    WHEN trip_distance < 2 THEN 'Short (< 2 mi)'
    WHEN trip_distance < 5 THEN 'Medium (2-5 mi)'
    WHEN trip_distance < 10 THEN 'Long (5-10 mi)'
    ELSE 'Very Long (10+ mi)'
  END as distance_category,
  COUNT(*) as trips,
  ROUND(AVG(fare_amount), 2) as avg_fare
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 AND trip_distance < 50
GROUP BY 1
ORDER BY trips DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Query 2.3: Daily Trip Volume
-- Purpose: Line Chart - Trips over time
-- Visualization: Create a LINE CHART with pickup_date on X-axis and trips on Y-axis
-- Save as: "Daily Trip Volume"

SELECT 
  DATE(tpep_pickup_datetime) as pickup_date,
  COUNT(*) as trips,
  ROUND(AVG(fare_amount), 2) as avg_fare,
  ROUND(SUM(fare_amount), 2) as total_revenue
FROM samples.nyctaxi.trips
WHERE tpep_pickup_datetime IS NOT NULL
GROUP BY DATE(tpep_pickup_datetime)
ORDER BY pickup_date
LIMIT 30;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Query 2.4: Distance vs Fare Analysis
-- Purpose: Scatter Plot - Relationship between distance and fare
-- Visualization: Create a SCATTER PLOT with trip_distance on X-axis and fare_amount on Y-axis
-- Save as: "Distance vs Fare"

SELECT 
  trip_distance,
  fare_amount
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 
  AND trip_distance < 20 
  AND fare_amount > 0
  AND fare_amount < 100
LIMIT 1000;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Query 2.5: Hourly Trip Pattern
-- Purpose: Bar Chart - Trips by hour of day
-- Visualization: Create a BAR CHART with pickup_hour on X-axis and trips on Y-axis
-- Save as: "Trips by Hour"

SELECT 
  HOUR(tpep_pickup_datetime) as pickup_hour,
  COUNT(*) as trips,
  ROUND(AVG(fare_amount), 2) as avg_fare
FROM samples.nyctaxi.trips
WHERE tpep_pickup_datetime IS NOT NULL
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY pickup_hour;

## Section 3: Dashboard Queries

Use these queries to build a complete **NYC Taxi Analytics Dashboard**:

**Dashboard Layout:**
- Row 1: Key Metrics (KPI cards)
- Row 2: Bar Chart (Distance Categories) + Line Chart (Daily Volume)
- Row 3: Scatter Plot (Distance vs Fare)
- Row 4: Table (Top Pickup Locations)

In [0]:
%sql
-- Dashboard Query 1: Key Metrics (KPI Cards)
-- Purpose: Summary statistics for dashboard header
-- Display as: 4 separate COUNTER visualizations

SELECT 
  COUNT(*) as total_trips,
  ROUND(AVG(trip_distance), 2) as avg_distance_miles,
  ROUND(AVG(fare_amount), 2) as avg_fare_usd,
  ROUND(SUM(fare_amount), 2) as total_revenue
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 AND fare_amount > 0;

In [0]:
%sql
-- Dashboard Query 2: Top Dropoff Locations
-- Purpose: Table - Most common destinations
-- Display as: TABLE

SELECT 
  dropoff_zip,
  COUNT(*) as trip_count,
  ROUND(AVG(fare_amount), 2) as avg_fare
FROM samples.nyctaxi.trips
WHERE dropoff_zip IS NOT NULL
GROUP BY dropoff_zip
ORDER BY trip_count DESC
LIMIT 10;

In [0]:
%sql
-- Dashboard Query 3: Trip Duration Analysis
-- Purpose: Bar Chart - Trips by duration
-- Visualization: Create a BAR CHART with duration_category on X-axis and trips on Y-axis

SELECT 
  CASE 
    WHEN TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) < 10 THEN 'Quick (< 10 min)'
    WHEN TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) < 20 THEN 'Short (10-20 min)'
    WHEN TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) < 40 THEN 'Medium (20-40 min)'
    ELSE 'Long (40+ min)'
  END as duration_category,
  COUNT(*) as trips
FROM samples.nyctaxi.trips
WHERE tpep_pickup_datetime IS NOT NULL 
  AND tpep_dropoff_datetime IS NOT NULL
  AND TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) > 0
  AND TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) < 120
GROUP BY 1
ORDER BY trips DESC;

## Section 4: Advanced Analysis (Optional)

These queries dive deeper into patterns and insights.

In [0]:
%sql
-- Query 4.1: Revenue by Day of Week
-- Purpose: Identify busiest days

SELECT 
  DAYOFWEEK(tpep_pickup_datetime) as day_of_week,
  CASE DAYOFWEEK(tpep_pickup_datetime)
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END as day_name,
  COUNT(*) as trips,
  ROUND(SUM(fare_amount), 2) as total_revenue
FROM samples.nyctaxi.trips
WHERE tpep_pickup_datetime IS NOT NULL
GROUP BY DAYOFWEEK(tpep_pickup_datetime), 2
ORDER BY day_of_week;

In [0]:
%sql
-- Query 4.2: Average Fare by Distance Range
-- Purpose: Understand pricing patterns

SELECT 
  FLOOR(trip_distance) as distance_mile,
  COUNT(*) as trips,
  ROUND(AVG(fare_amount), 2) as avg_fare,
  ROUND(MIN(fare_amount), 2) as min_fare,
  ROUND(MAX(fare_amount), 2) as max_fare
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 
  AND trip_distance < 20
  AND fare_amount > 0
GROUP BY FLOOR(trip_distance)
ORDER BY distance_mile;

In [0]:
%sql
-- Query 4.3: Popular Routes (Pickup to Dropoff)
-- Purpose: Identify common travel patterns

SELECT 
  pickup_zip,
  dropoff_zip,
  COUNT(*) as trip_count,
  ROUND(AVG(trip_distance), 2) as avg_distance,
  ROUND(AVG(fare_amount), 2) as avg_fare
FROM samples.nyctaxi.trips
WHERE pickup_zip IS NOT NULL 
  AND dropoff_zip IS NOT NULL
  AND pickup_zip != dropoff_zip
GROUP BY pickup_zip, dropoff_zip
ORDER BY trip_count DESC
LIMIT 20;

## Section 5: Data Quality Checks (Optional)

Validate data before creating dashboards.

In [0]:
%sql
-- Query 5.1: Check for NULL values

SELECT 
  COUNT(*) as total_rows,
  SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) as null_pickup_datetime,
  SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) as null_dropoff_datetime,
  SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) as null_distance,
  SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) as null_fare,
  SUM(CASE WHEN pickup_zip IS NULL THEN 1 ELSE 0 END) as null_pickup_zip,
  SUM(CASE WHEN dropoff_zip IS NULL THEN 1 ELSE 0 END) as null_dropoff_zip
FROM samples.nyctaxi.trips;

In [0]:
%sql
-- Query 5.2: Check for outliers

SELECT 
  'Very Short Distance' as category,
  COUNT(*) as count
FROM samples.nyctaxi.trips
WHERE trip_distance < 0.1
UNION ALL
SELECT 
  'Very Long Distance',
  COUNT(*)
FROM samples.nyctaxi.trips
WHERE trip_distance > 50
UNION ALL
SELECT 
  'Very Low Fare',
  COUNT(*)
FROM samples.nyctaxi.trips
WHERE fare_amount < 2.5
UNION ALL
SELECT 
  'Very High Fare',
  COUNT(*)
FROM samples.nyctaxi.trips
WHERE fare_amount > 200;

## Next Steps

**You've completed the training queries!**

### To Create Visualizations:
1. Run any query above
2. Click the "+" button below results
3. Select "Visualization"
4. Choose chart type (Bar, Line, Scatter, Table)
5. Configure axes and title
6. Save visualization

### To Build a Dashboard:
1. Go to "Dashboards" in sidebar
2. Create new dashboard
3. Add saved visualizations
4. Arrange layout
5. Publish and share!
